# Aegis Command - Time-Based Contiguous Holdout 5-Fold Fine-Tuning Pipeline

This notebook trains and evaluates the **Time-Based Contiguous Holdout 5-Fold Fine-Tuning Protocol** (50 Epochs, imgsz 1280px) on Google Colab GPU.

---

### CRITICAL METHODOLOGICAL DISCLOSURE: SAME-SCENE PERFORMANCE (SINGLE SOURCE VIDEO)
> **EVALUATION SCOPE & LIMITATIONS**:
> All 155 human-verified ground-truth frames (1,948 labeled objects) originate from a single tactical UAV reconnaissance flight (`uav-alpha/video.mp4`).
>
> While this protocol enforces **strict contiguous time partitioning** and **4-frame temporal purge buffers** (eliminating frame-to-frame temporal leakage and optical autocorrelation across train and test folds), all folds share the same sensor elevation, sensor optical characteristics, illumination, and physical AOI (Area of Interest).
>
> Consequently, all reported metrics measure **same-scene holdout performance**. This benchmark demonstrates in-domain detection reliability and scale recovery, but **does NOT claim cross-domain or out-of-distribution generalization** across unseen operational environments or different camera sensors.

---

### Strict Contiguous 5-Fold Partitioning (Purge Buffer = 4 frames):
- **155 frames** partitioned into 5 temporal blocks of 31 frames:
  - **Fold 1**: Val `0..30` (31 frames), Purge Buffer `31..34` (4 frames), Train `35..154` (120 frames, 77.4%)
  - **Fold 2**: Train `0..26` (27 frames), Purge Buffer `27..30` (4 frames), Val `31..61` (31 frames), Purge Buffer `62..65` (4 frames), Train `66..154` (89 frames) -> Train total: 116 frames (74.8%)
  - **Fold 3**: Train `0..57`, Purge Buffer `58..61`, Val `62..92` (31 frames), Purge Buffer `93..96`, Train `97..154` -> Train total: 116 frames (74.8%)
  - **Fold 4**: Train `0..88`, Purge Buffer `89..92`, Val `93..123` (31 frames), Purge Buffer `124..127`, Train `128..154` -> Train total: 116 frames (74.8%)
  - **Fold 5**: Train `0..119` (120 frames, 77.4%), Purge Buffer `120..123` (4 frames), Val `124..154` (31 frames)
- **Zero Temporal Leakage**: Validation sets form a partition of the 155 frames (all 155 frames evaluated as val in exactly one fold; pairwise disjoint).

---

### Weight Handling Policy:
- Trained weights are **checkpointed directly to Google Drive** (`/content/drive/MyDrive/aegis_checkpoints/`).
- Weights are **NEVER committed to git** (per repository zero-binary policy).
- The cryptographic SHA-256 hash of each fold's `best.pt` is recorded in run metadata alongside training hyper-parameters.

In [ ]:
# 1. Verify GPU Allocation
!nvidia-smi

In [ ]:
# 2. Install Required Dependencies
%pip install -q ultralytics albumentations opencv-python pandas tabulate numpy

In [ ]:
# 3. Clone Repository & Checkout Branch feat/detection-v3
!git clone https://github.com/TanmayJain-dev/Aegis-Command.git
%cd Aegis-Command
!git checkout feat/detection-v3
!git pull

In [ ]:
# 4. Cryptographic Certification Gate Validation
from pathlib import Path
from vision.eval.eval_detection import check_verification_gate

marker = check_verification_gate(Path("vision/eval/labels"))
print(f"[GATE PASSED] Cryptographically verified ground truth at {marker}")

In [ ]:
# 5. Mount Google Drive for Weight Checkpointing (No weights committed to git)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/aegis_checkpoints')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Google Drive checkpoint directory ready at: {DRIVE_DIR}")

In [ ]:
# 6. Execute 5-Fold Contiguous Holdout Fine-Tuning (50 Epochs, imgsz 1280px)
import hashlib
import json
import shutil
from ultralytics import YOLO
from vision.eval.split_dataset import split_dataset
from vision.eval.eval_detection import run_evaluation

# Protocol Hyperparameters (Starting from best bakeoff config: yolov8m at 1280px)
TRAIN_CONFIG = {
    "base_model": "yolov8m.pt",
    "epochs": 50,
    "imgsz": 1280,
    "batch": 8,
    "lr0": 0.01,
    "lrf": 0.01,
    "mosaic": 1.0,
    "mixup": 0.15,
    "box": 7.5,
    "cls": 0.5,
    "temporal_purge_buffer_frames": 4,
    "eval_scope": "same-scene performance (single source video)"
}

fold_results = []

for fold in range(1, 6):
    print(f"\n{'='*75}")
    print(f"STARTING FOLD {fold} / 5 (50 Epochs, {TRAIN_CONFIG['base_model']} @ {TRAIN_CONFIG['imgsz']}px)")
    print(f"{'='*75}")
    
    # Partition dataset with 4-frame purge buffer
    split_manifest = split_dataset(fold=fold, num_folds=5, buffer_frames=TRAIN_CONFIG["temporal_purge_buffer_frames"])
    
    # Initialize backbone from best bakeoff model
    model = YOLO(TRAIN_CONFIG["base_model"])
    
    # Train fine-tuned model
    train_run = model.train(
        data="vision/data/aegis_data.yaml",
        epochs=TRAIN_CONFIG["epochs"],
        imgsz=TRAIN_CONFIG["imgsz"],
        batch=TRAIN_CONFIG["batch"],
        device=0,
        workers=4,
        lr0=TRAIN_CONFIG["lr0"],
        lrf=TRAIN_CONFIG["lrf"],
        mosaic=TRAIN_CONFIG["mosaic"],
        mixup=TRAIN_CONFIG["mixup"],
        box=TRAIN_CONFIG["box"],
        cls=TRAIN_CONFIG["cls"],
        project="runs/train",
        name=f"aegis_fold{fold}",
        save=True,
        plots=True
    )
    
    # Checkpoint best weights to Drive and record cryptographic SHA-256
    best_weights_path = Path(f"runs/train/aegis_fold{fold}/weights/best.pt")
    weight_sha256 = "N/A"
    if best_weights_path.exists():
        weight_bytes = best_weights_path.read_bytes()
        weight_sha256 = hashlib.sha256(weight_bytes).hexdigest()
        drive_dst = DRIVE_DIR / f"aegis_5fold_fold{fold}_best.pt"
        shutil.copy(best_weights_path, drive_dst)
        print(f"[CHECKPOINT] Copied Fold {fold} weights to Drive: {drive_dst}")
        print(f"[SHA-256] {weight_sha256}")
    
    # Evaluate Fine-Tuned Model on Holdout Val
    run_id_ft = f"yolov8m_1280px_val_fold{fold}_finetuned"
    eval_out_ft = f"docs/results/raw/{run_id_ft}/eval_results.json"
    run_evaluation(
        model_path=str(best_weights_path),
        split="val",
        fold=fold,
        imgsz=TRAIN_CONFIG["imgsz"],
        device="cuda:0",
        native_classes=True,
        save_raw=True,
        raw_root="docs/results/raw",
        run_id=run_id_ft,
        out_results=eval_out_ft
    )
    
    # Evaluate Pre-Trained Baseline on Same Holdout Val
    run_id_base = f"yolov8m_1280px_val_fold{fold}_baseline"
    eval_out_base = f"docs/results/raw/{run_id_base}/eval_results.json"
    run_evaluation(
        model_path=TRAIN_CONFIG["base_model"],
        split="val",
        fold=fold,
        imgsz=TRAIN_CONFIG["imgsz"],
        device="cuda:0",
        native_classes=False,
        save_raw=True,
        raw_root="docs/results/raw",
        run_id=run_id_base,
        out_results=eval_out_base
    )
    
    # Record fold metrics
    with open(eval_out_ft, "r", encoding="utf-8") as f:
        ft_data = json.load(f)
    with open(eval_out_base, "r", encoding="utf-8") as f:
        base_data = json.load(f)
        
    fold_results.append({
        "fold": fold,
        "train_count": split_manifest[f"fold_{fold}"]["train_count"],
        "buffer_count": split_manifest[f"fold_{fold}"]["buffer_count"],
        "val_count": split_manifest[f"fold_{fold}"]["val_count"],
        "weight_sha256": weight_sha256,
        "training_args": TRAIN_CONFIG,
        "finetuned_metrics": ft_data.get("metrics", {}),
        "baseline_metrics": base_data.get("metrics", {})
    })

# Save full experiment record
summary_record_path = Path("docs/results/raw/task4_5fold_finetune_summary.json")
summary_record_path.parent.mkdir(parents=True, exist_ok=True)
with open(summary_record_path, "w", encoding="utf-8") as f:
    json.dump({
        "protocol": "5-fold contiguous holdout cross-validation",
        "evaluation_scope": "same-scene performance (single source video)",
        "train_config": TRAIN_CONFIG,
        "folds": fold_results
    }, f, indent=2)
print(f"\n[SAVED] 5-Fold summary saved to {summary_record_path}")

In [ ]:
# 7. Generate Per-Fold and Aggregate (Mean +/- Std) Results Table
import numpy as np
import pandas as pd
from tabulate import tabulate

print("=" * 90)
print("EVALUATION: same-scene performance (single source video)")
print("5-Fold Contiguous Holdout Fine-Tuning (yolov8m @ 1280px, 50 Epochs, Purge Buffer=4)")
print("=" * 90)

rows = []
p_ap50s, t_ap50s, m_ap50s, map_50_95s = [], [], [], []

for f in fold_results:
    f_num = f["fold"]
    ft_m = f["finetuned_metrics"]
    base_m = f["baseline_metrics"]
    
    p_ap50 = ft_m.get("per_class", {}).get("person", {}).get("ap50", 0.0)
    t_ap50 = ft_m.get("per_class", {}).get("truck", {}).get("ap50", 0.0)
    overall_ap50 = ft_m.get("overall", {}).get("map50", ft_m.get("mAP50", 0.0))
    overall_map = ft_m.get("overall", {}).get("map50_95", ft_m.get("mAP50_95", 0.0))
    best_f1 = ft_m.get("overall", {}).get("best_f1", ft_m.get("f1_score", 0.0))
    
    p_ap50s.append(p_ap50)
    t_ap50s.append(t_ap50)
    m_ap50s.append(overall_ap50)
    map_50_95s.append(overall_map)
    
    rows.append([
        f"Fold {f_num}",
        f"{f['train_count']} / {f['buffer_count']} / {f['val_count']}",
        f"{p_ap50:.4f}",
        f"{t_ap50:.4f}",
        f"{overall_ap50:.4f}",
        f"{overall_map:.4f}",
        f"{best_f1:.4f}",
        f["weight_sha256"][:12] + "..."
    ])

# Compute Aggregate Mean +/- Std
mean_row = [
    "MEAN +/- STD",
    "117.6 / 6.4 / 31.0",
    f"{np.mean(p_ap50s):.4f} +/- {np.std(p_ap50s):.4f}",
    f"{np.mean(t_ap50s):.4f} +/- {np.std(t_ap50s):.4f}",
    f"{np.mean(m_ap50s):.4f} +/- {np.std(m_ap50s):.4f}",
    f"{np.mean(map_50_95s):.4f} +/- {np.std(map_50_95s):.4f}",
    "-",
    "-"
]
rows.append(mean_row)

headers = ["Fold", "Train/Buf/Val", "Person AP50", "Truck AP50", "mAP50", "mAP50-95", "Best F1", "Weights SHA-256"]
print(tabulate(rows, headers=headers, tablefmt="grid"))

# Print Training Args and Weight Checkpoint Ledger
print("\n" + "=" * 90)
print("TRAINING ARGS & CHECKPOINT LEDGER (Zero-binary Git Policy)")
print("=" * 90)
ledger_rows = []
for f in fold_results:
    ledger_rows.append([
        f"Fold {f['fold']}",
        f["training_args"]["base_model"],
        f["training_args"]["imgsz"],
        f["training_args"]["epochs"],
        f["training_args"]["batch"],
        f["training_args"]["lr0"],
        f["weight_sha256"]
    ])
ledger_headers = ["Fold", "Base Model", "imgsz", "Epochs", "Batch", "lr0", "Full Weight SHA-256"]
print(tabulate(ledger_rows, headers=ledger_headers, tablefmt="grid"))

In [ ]:
# 8. Package Raw Prediction Dumps for Verification
!zip -r colab_train_5fold_raw.zip docs/results/raw/yolov8m_* docs/results/raw/task4_5fold_finetune_summary.json

from google.colab import files
files.download('colab_train_5fold_raw.zip')
print("\n[DOWNLOAD READY] Extract colab_train_5fold_raw.zip into repo docs/results/raw/")